In [ ]:
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


# Qwen 0 ATS float16

In [ ]:
# ============================================================
# SCRIPT: Evaluation Qwen 2.5 7B (0-shot) — Metrics + "OOF" saves
# VERSION bf16 — aligned with the few-shot CV script
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = "DATA/outputs/predictions/zeroshot_llms"
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.environ["HF_TOKEN"]

# Aligned with the CV script
BATCH_SIZE = 4
MAX_NEW_TOKENS = 20
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading data and building the Gold Standard from {path}...")
    df = pd.read_excel(path)

    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column in the input file: {c}")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valid: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valid: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valid    : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (Qwen bf16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen_Pred", "Qwen_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    def get_status(row):
        g, p = row["y_true_target"], row["Qwen_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bf16...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Loading in bf16 (like the CV script)
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )
    model.eval()

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["Qwen_Pred"] = preds
    df["Qwen_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (Qwen bf16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Confusion matrix: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_qwen_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen_Pred", "Qwen_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_qwen_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Done] Saved files:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDirectory: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# LLama 0 shot ATS 16bits

In [ ]:
# ============================================================
# SCRIPT: Evaluation Llama 3.1 8B (0-shot) — Metrics + "OOF" saves
# (Here "OOF" = row-by-row predictions over the whole dataset, without folds)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Saved files:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (all-in-one)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = "DATA/outputs/predictions/zeroshot_llms"
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

HF_TOKEN = os.environ["HF_TOKEN"]

# Aligned with script 1
BATCH_SIZE = 4
MAX_NEW_TOKENS = 20
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading data and building the Gold Standard from {path}...")
    df = pd.read_excel(path)

    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column in the input file: {c}")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valid: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valid: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valid    : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (Llama 3.1 bf16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()

    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Llama_Pred", "Llama_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    def get_status(row):
        g, p = row["y_true_target"], row["Llama_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bf16...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Loading in bf16 (like script 1)
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model.eval()

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)

    df["Llama_Pred"] = preds
    df["Llama_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (Llama 3.1 bf16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Confusion matrix: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_llama_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Llama_Pred", "Llama_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_llama_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Done] Saved files:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDirectory: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Mistral Nemo 0 shot

In [ ]:
# ============================================================
# SCRIPT: Evaluation Mistral-Nemo 12B (0-shot) — Metrics + "OOF" saves
# VERSION fp32/bf16 depending on available GPU (A100)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = "DATA/outputs/predictions/zeroshot_llms"
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Mistral-Nemo 12B Instruct (Mistral's "14B")
MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"
HF_TOKEN = os.environ["HF_TOKEN"]

BATCH_SIZE = 4
MAX_NEW_TOKENS = 20
MAX_PROMPT_LEN = 4096

# Precision choice: "fp32" for A100-80GB, "bf16" for A100-40GB
PRECISION = "bf16"  # Switch to "fp32" if you have 80 GB

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def get_gpu_memory():
    if torch.cuda.is_available():
        return torch.cuda.get_device_properties(0).total_memory / 1e9
    return 0

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading data and building the Gold Standard from {path}...")
    df = pd.read_excel(path)

    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column in the input file: {c}")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valid: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valid: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valid    : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (Mistral {PRECISION}) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Mistral_Pred", "Mistral_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    def get_status(row):
        g, p = row["y_true_target"], row["Mistral_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    # Automatic GPU detection
    gpu_mem = get_gpu_memory()
    print(f"\n[GPU] Detected VRAM: {gpu_mem:.1f} GB")

    # dtype selection
    if PRECISION == "fp32":
        dtype = torch.float32
        precision_str = "fp32"
        print(f"[Init] Loading in float32 (~48 GB VRAM)")
    else:
        dtype = torch.bfloat16
        precision_str = "bf16"
        print(f"[Init] Loading in bfloat16 (~24 GB VRAM)")

    print(f"[Init] Loading model {MODEL_NAME}...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model.eval()

    print(f"    ✓ Model loaded in {precision_str}")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["Mistral_Pred"] = preds
    df["Mistral_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print(f"\n=== RESULTS (Mistral {precision_str}) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Confusion matrix: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_mistral_{precision_str}_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Mistral_Pred", "Mistral_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_mistral_{precision_str}_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Done] Saved files:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDirectory: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Qwen 32 B

In [ ]:
# ============================================================
# SCRIPT: Evaluation Qwen 2.5 32B (0-shot) — Metrics + "OOF" saves
# VERSION: bfloat16 (half precision, no quantization)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Saved files:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (all-in-one)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = "DATA/outputs/predictions/zeroshot_llms"
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"
HF_TOKEN = os.environ["HF_TOKEN"]

# Batch size for 32B in bfloat16 (~64 GB VRAM)
BATCH_SIZE = 2
MAX_NEW_TOKENS = 64
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    # robust regex
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading data and building the Gold Standard from {path}...")
    df = pd.read_excel(path)

    # minimal columns
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column in the input file: {c}")

    # annotation columns (NaN if absent)
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    # GoldDerived = agreement A2/A1, otherwise A3
    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # filter texts
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valid: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valid: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valid    : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (32-bit) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # decode per item (slice the generated part)
        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    # ignore invalid preds + missing labels
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    # y_true_col already contains labels 0/1/NaN
    # keep only key cols
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen_Pred", "Qwen_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    # error type vs target
    def get_status(row):
        g, p = row["y_true_target"], row["Qwen_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bfloat16...")
    print("       ~64 GB VRAM for a 32B model in BF16")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Loading in bfloat16 (half precision, no quantization)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,  # bfloat16 precision
        device_map="auto",
        trust_remote_code=True
    )

    print(f"       Model loaded. dtype: {model.dtype}")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["Qwen_Pred"] = preds
    df["Qwen_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    # Print summary + confusion matrices
    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (bfloat16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Confusion matrix: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen_Pred", "Qwen_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target (3 files)
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Done] Saved files:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDirectory: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Gemma 27 B

In [ ]:
# ============================================================
# SCRIPT: Evaluation Gemma 27B (0-shot) — Metrics + "OOF" saves
# VERSION: bfloat16 (half precision, no quantization)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Saved files:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (all-in-one)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = "DATA/outputs/predictions/zeroshot_llms"
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "google/gemma-2-27b-it"
HF_TOKEN = os.environ["HF_TOKEN"]

# Batch size for 27B in bfloat16 (~54 GB VRAM)
BATCH_SIZE = 2
MAX_NEW_TOKENS = 64
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    # robust regex
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading data and building the Gold Standard from {path}...")
    df = pd.read_excel(path)

    # minimal columns
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column in the input file: {c}")

    # annotation columns (NaN if absent)
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    # GoldDerived = agreement A2/A1, otherwise A3
    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # filter texts
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valid: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valid: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valid    : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    # Gemma 2 uses a chat format with <start_of_turn> and <end_of_turn>
    # But with apply_chat_template we use the standard format
    messages = [
        {"role": "user", "content": f"""{GRILLE_ANNOTATION}

Article: {str(row['article_text'])}
Extrait: {str(row['text'])}

Réponse (oui/non) :"""}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (bfloat16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # decode per item (slice the generated part)
        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    # ignore invalid preds + missing labels
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    # y_true_col already contains labels 0/1/NaN
    # keep only key cols
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Gemma_Pred", "Gemma_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    # error type vs target
    def get_status(row):
        g, p = row["y_true_target"], row["Gemma_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bfloat16...")
    print("       ~54 GB VRAM for a 27B model in BF16")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Loading in bfloat16 (half precision, no quantization)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,  # bfloat16 precision
        device_map="auto",
        trust_remote_code=True
    )

    print(f"       Model loaded. dtype: {model.dtype}")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["Gemma_Pred"] = preds
    df["Gemma_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    # Print summary + confusion matrices
    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (bfloat16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Confusion matrix: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Gemma_Pred", "Gemma_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target (3 files)
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Done] Saved files:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDirectory: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Command R 35B 0 shot

In [ ]:
# ============================================================
# SCRIPT: Evaluation Command R (0-shot) — Metrics + "OOF" saves
# VERSION: bfloat16 (half precision, no quantization)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Saved files:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (all-in-one)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = "DATA/outputs/predictions/zeroshot_llms"
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "CohereForAI/c4ai-command-r-v01"
HF_TOKEN = os.environ["HF_TOKEN"]

# Batch size for Command R 35B in bfloat16 (~70 GB VRAM)
BATCH_SIZE = 2
MAX_NEW_TOKENS = 64
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    # robust regex
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading data and building the Gold Standard from {path}...")
    df = pd.read_excel(path)

    # minimal columns
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column in the input file: {c}")

    # annotation columns (NaN if absent)
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    # GoldDerived = agreement A2/A1, otherwise A3
    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # filter texts
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valid: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valid: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valid    : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    # Command R supports the standard format with system + user
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (bfloat16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # decode per item (slice the generated part)
        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    # ignore invalid preds + missing labels
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    # y_true_col already contains labels 0/1/NaN
    # keep only key cols
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "CommandR_Pred", "CommandR_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    # error type vs target
    def get_status(row):
        g, p = row["y_true_target"], row["CommandR_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bfloat16...")
    print("       ~70 GB VRAM for Command R 35B in BF16")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Loading in bfloat16 (half precision, no quantization)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,  # bfloat16 precision
        device_map="auto",
        trust_remote_code=True
    )

    print(f"       Model loaded. dtype: {model.dtype}")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["CommandR_Pred"] = preds
    df["CommandR_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    # Print summary + confusion matrices
    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (bfloat16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Confusion matrix: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "CommandR_Pred", "CommandR_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target (3 files)
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Done] Saved files:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDirectory: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Aya 32B 0 shot

In [ ]:
# ============================================================
# SCRIPT: Evaluation Aya Expanse 32B (0-shot) — Metrics + "OOF" saves
# VERSION: bfloat16 (half precision, no quantization)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Saved files:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (all-in-one)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = "DATA/outputs/predictions/zeroshot_llms"
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "CohereForAI/aya-expanse-32b"
HF_TOKEN = os.environ["HF_TOKEN"]

# Batch size for Aya Expanse 32B in bfloat16 (~64 GB VRAM)
BATCH_SIZE = 2
MAX_NEW_TOKENS = 64
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    # robust regex
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading data and building the Gold Standard from {path}...")
    df = pd.read_excel(path)

    # minimal columns
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column in the input file: {c}")

    # annotation columns (NaN if absent)
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    # GoldDerived = agreement A2/A1, otherwise A3
    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # filter texts
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valid: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valid: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valid    : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    # Aya Expanse uses the Cohere format with system + user
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (bfloat16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # decode per item (slice the generated part)
        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    # ignore invalid preds + missing labels
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    # y_true_col already contains labels 0/1/NaN
    # keep only key cols
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "AyaExpanse_Pred", "AyaExpanse_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    # error type vs target
    def get_status(row):
        g, p = row["y_true_target"], row["AyaExpanse_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bfloat16...")
    print("       ~64 GB VRAM for Aya Expanse 32B in BF16")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Loading in bfloat16 (half precision, no quantization)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,  # bfloat16 precision
        device_map="auto",
        trust_remote_code=True
    )

    print(f"       Model loaded. dtype: {model.dtype}")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["AyaExpanse_Pred"] = preds
    df["AyaExpanse_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    # Print summary + confusion matrices
    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (bfloat16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Confusion matrix: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "AyaExpanse_Pred", "AyaExpanse_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target (3 files)
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Done] Saved files:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDirectory: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# LlaMa 70B 4bits

In [ ]:
# ============================================================
# SCRIPT: Evaluation Llama 3.1 70B (0-shot) — 4-bit quantization
# Uses bitsandbytes for 4-bit quantization (QLoRA-style)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Saved files:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (all-in-one)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = "DATA/outputs/predictions/zeroshot_llms"
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "meta-llama/Llama-3.1-70B-Instruct"

HF_TOKEN = os.environ["HF_TOKEN"]

# Reduced batch size for the 70B (even in 4-bit the model is more memory-hungry)
BATCH_SIZE = 2
MAX_NEW_TOKENS = 20
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading data and building the Gold Standard from {path}...")
    df = pd.read_excel(path)

    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column in the input file: {c}")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valid: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valid: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valid    : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (Llama 3.1 70B 4-bit) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()

    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Llama_Pred", "Llama_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    def get_status(row):
        g, p = row["y_true_target"], row["Llama_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in 4-bit (bitsandbytes)...")

    # bitsandbytes configuration for 4-bit quantization (NF4)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",              # NormalFloat4 - better quality
        bnb_4bit_compute_dtype=torch.bfloat16,  # Compute in bf16 for speed
        bnb_4bit_use_double_quant=True,         # Double quantization to save memory
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    # Print GPU memory usage
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"    GPU Memory: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)

    df["Llama_Pred"] = preds
    df["Llama_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (Llama 3.1 70B 4-bit) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Confusion matrix: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_llama70b_4bit_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Llama_Pred", "Llama_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_llama70b_4bit_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Done] Saved files:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDirectory: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Saul 7B fp16

In [ ]:
# ============================================================
# SCRIPT: Evaluation Saul 7B Instruct (0-shot) — Metrics + "OOF" saves
# (Here "OOF" = row-by-row predictions over the whole dataset, without folds)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Saved files:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (all-in-one)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = "DATA/outputs/predictions/zeroshot_llms"
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Saul-7B-Instruct (legal model based on Mistral)
MODEL_NAME = "Equall/Saul-7B-Instruct-v1"

HF_TOKEN = os.environ["HF_TOKEN"]

BATCH_SIZE = 4
MAX_NEW_TOKENS = 20
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading data and building the Gold Standard from {path}...")
    df = pd.read_excel(path)

    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column in the input file: {c}")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valid: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valid: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valid    : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    """
    Saul-7B-Instruct (based on Mistral) does NOT accept "system" messages.
    So the system instructions are folded directly into the user message.
    """
    user_content = f"""{GRILLE_ANNOTATION}

Article: {str(row['article_text'])}
Extrait: {str(row['text'])}

Réponse (oui/non) :"""

    messages = [
        {"role": "user", "content": user_content}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (Saul-7B-Instruct bf16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()

    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Saul_Pred", "Saul_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    def get_status(row):
        g, p = row["y_true_target"], row["Saul_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bf16...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Loading in bf16
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model.eval()

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)

    df["Saul_Pred"] = preds
    df["Saul_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (Saul-7B-Instruct bf16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Confusion matrix: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_saul_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Saul_Pred", "Saul_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_saul_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Done] Saved files:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDirectory: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Reasoning

In [ ]:
# ============================================================
# SCRIPT: Evaluation Qwen3-32B (Thinking Mode) — Metrics + "OOF" saves
# VERSION: bfloat16 (half precision, no quantization)
#
# With enable_thinking=True, Qwen3-32B emits
# <think>...</think> content before its final answer.
#
# IMPORTANT: do not use greedy decoding in thinking mode!
#            Use temperature=0.6, top_p=0.95, top_k=20
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Saved files:
#   - metrics_extended_*.xlsx
#   - OOF_thinking_A1_*.xlsx
#   - OOF_thinking_A2_*.xlsx
#   - OOF_thinking_Gold_*.xlsx
#   - predictions_full_*.xlsx (all-in-one, with thinking)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = "DATA/outputs/predictions/zeroshot_llms"
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Qwen3-32B - latest model with native thinking mode
MODEL_NAME = "Qwen/Qwen3-32B"
HF_TOKEN = os.environ["HF_TOKEN"]

# Reduced batch size because thinking generates many more tokens
BATCH_SIZE = 1  # Safer for thinking (long generation)

# High MAX_NEW_TOKENS for thinking (recommended: up to 32768)
# Can be capped at ~8192 for our task
MAX_NEW_TOKENS = 8192
MAX_PROMPT_LEN = 4096

# Generation parameters for Thinking Mode (MANDATORY: no greedy!)
# Source: https://huggingface.co/Qwen/Qwen3-32B
GENERATION_CONFIG = {
    "do_sample": True,          # MANDATORY in thinking mode
    "temperature": 0.6,         # Recommended by Qwen
    "top_p": 0.95,              # Recommended by Qwen
    "top_k": 20,                # Recommended by Qwen
    "min_p": 0.0,               # Recommended by Qwen
}

# Prompt adapted to encourage legal reasoning
GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

Analyse l'extrait en considérant:
1. Le raisonnement juridique utilisé par le juge
2. Les concepts juridiques mobilisés (même sans citation explicite)
3. La cohérence entre l'article proposé et le raisonnement de la décision

À la fin de ton analyse, conclus OBLIGATOIREMENT par une ligne contenant uniquement:
RÉPONSE: oui
ou
RÉPONSE: non

- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique par exemple)
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    # robust regex
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading data and building the Gold Standard from {path}...")
    df = pd.read_excel(path)

    # minimal columns
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column in the input file: {c}")

    # annotation columns (NaN if absent)
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    # GoldDerived = agreement A2/A1, otherwise A3
    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # filter texts
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valid: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valid: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valid    : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_thinking(row):
    """
    Build the prompt for Qwen3-32B in thinking mode.
    """
    user_content = f"""{GRILLE_ANNOTATION}

Article du Code civil:
{str(row['article_text'])}

Extrait de la décision de justice:
{str(row['text'])}

Analyse cet extrait et détermine si l'article est implicitement appliqué."""

    messages = [
        {"role": "user", "content": user_content}
    ]
    return messages

def extract_thinking_and_answer(response_text, output_ids=None, tokenizer=None):
    """
    Extract the thinking content and the final answer.

    Method 1 (preferred): use token ID 151668 (</think>)
    Method 2 (fallback): regex on <think>...</think>
    """
    response = str(response_text)

    # Method 1: parse via token ID if available
    if output_ids is not None and tokenizer is not None:
        try:
            # Token ID 151668 = </think>
            output_list = output_ids.tolist() if hasattr(output_ids, 'tolist') else list(output_ids)
            # Find the index of </think> (151668) from the end
            try:
                index = len(output_list) - output_list[::-1].index(151668)
                thinking_content = tokenizer.decode(output_list[:index], skip_special_tokens=True).strip()
                final_answer = tokenizer.decode(output_list[index:], skip_special_tokens=True).strip()
                return thinking_content, final_answer
            except ValueError:
                # </think> token not found, continue with method 2
                pass
        except Exception:
            pass

    # Method 2: regex fallback
    think_match = re.search(r'<think>(.*?)</think>', response, re.DOTALL)
    thinking_content = think_match.group(1).strip() if think_match else ""

    # Extract the answer after </think>
    if '</think>' in response:
        final_answer = response.split('</think>')[-1].strip()
    else:
        # If no think tag, the whole response is used
        final_answer = response

    return thinking_content, final_answer

def parse_response(response_text, output_ids=None, tokenizer=None):
    """
    Parse the response to extract oui/non.
    First looks for "RÉPONSE: oui/non", otherwise falls back to simple detection.
    """
    _, final_answer = extract_thinking_and_answer(response_text, output_ids, tokenizer)
    text = final_answer.lower().strip()

    # Priority pattern: "RÉPONSE: oui/non"
    reponse_match = re.search(r'réponse\s*:\s*(oui|non)', text)
    if reponse_match:
        return 1 if reponse_match.group(1) == 'oui' else 0

    # Fallback: look for oui/non in the final answer
    text_clean = re.sub(r"[^\w\s]", "", text)

    if text_clean.startswith("oui") or text_clean == "oui": return 1
    if text_clean.startswith("non") or text_clean == "non": return 0

    words = text_clean.split()
    # Take the last words (more likely to be the conclusion)
    last_words = words[-10:] if len(words) > 10 else words

    if "oui" in last_words and "non" not in last_words: return 1
    if "non" in last_words and "oui" not in last_words: return 0

    # Last fallback over the whole response
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0

    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] Thinking-mode inference on {len(df)} rows...")
    print(f"    MAX_NEW_TOKENS: {MAX_NEW_TOKENS}")
    print(f"    BATCH_SIZE: {BATCH_SIZE}")
    print(f"    Generation config: {GENERATION_CONFIG}")

    all_preds = []
    all_raw = []
    all_thinking = []

    # Prepare prompts with enable_thinking=True
    prompts_formatted = []
    for _, row in df.iterrows():
        messages = build_prompt_thinking(row)
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True  # Enable thinking mode
        )
        prompts_formatted.append(prompt)

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Thinking Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                **GENERATION_CONFIG  # Sampling parameters (not greedy!)
            )

        # Decode per item
        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            output_ids = outputs[j][prompt_len:]
            response = tokenizer.decode(output_ids, skip_special_tokens=True)

            thinking, final = extract_thinking_and_answer(response, output_ids, tokenizer)
            pred = parse_response(response, output_ids, tokenizer)

            all_raw.append(response)
            all_thinking.append(thinking)
            all_preds.append(pred)

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw, all_thinking

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    # ignore invalid preds + missing labels
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen3_Pred", "Qwen3_Raw", "Qwen3_Thinking"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    def get_status(row):
        g, p = row["y_true_target"], row["Qwen3_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_thinking_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# THINKING ANALYSIS (bonus)
# ============================================================
def analyze_thinking(df, timestamp):
    """
    Qualitative analysis of the 'thinking' content generated by Qwen3.
    Useful to understand the model's reasoning.
    """
    stats = {
        "avg_thinking_length": df["Qwen3_Thinking"].apply(len).mean(),
        "median_thinking_length": df["Qwen3_Thinking"].apply(len).median(),
        "empty_thinking_count": (df["Qwen3_Thinking"] == "").sum(),
        "avg_words_thinking": df["Qwen3_Thinking"].apply(lambda x: len(x.split())).mean(),
    }

    print("\n=== THINKING ANALYSIS ===")
    print(f"    Mean thinking length: {stats['avg_thinking_length']:.0f} chars")
    print(f"    Median thinking length: {stats['median_thinking_length']:.0f} chars")
    print(f"    Empty thinking: {stats['empty_thinking_count']} cases")
    print(f"    Mean thinking words: {stats['avg_words_thinking']:.0f}")

    # Save a few thinking examples
    sample_df = df[["decision_id", "Qwen3_Pred", "Qwen3_Thinking"]].head(10)
    sample_path = os.path.join(OUTPUT_PATH, f"thinking_samples_{timestamp}.xlsx")
    sample_df.to_excel(sample_path, index=False)

    return stats, sample_path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"\n[Init] Loading model {MODEL_NAME} in bfloat16...")
    print("       Qwen3-32B: latest model with native thinking mode")
    print("       Requires transformers>=4.51.0")
    print("       ~64 GB VRAM for a 32B model in BF16")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Loading in bfloat16 (half precision, no quantization)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    print(f"       Model loaded. dtype: {model.dtype}")

    # Inference with thinking
    preds, raw_texts, thinking_texts = run_inference(df, model, tokenizer)
    df["Qwen3_Pred"] = preds
    df["Qwen3_Raw"] = raw_texts
    df["Qwen3_Thinking"] = thinking_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    # Print summary + confusion matrices
    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS Qwen3-32B (Thinking Mode) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Confusion matrix: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_thinking_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Thinking analysis
    thinking_stats, sample_path = analyze_thinking(df, timestamp)

    # Save full predictions (with thinking)
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen3_Pred", "Qwen3_Raw", "Qwen3_Thinking"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_thinking_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target (3 files)
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Done] Saved files:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- Thinking samples: {sample_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDirectory: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# GOLD - class + metrics


In [ ]:
"""
F1 and accuracy for 9 LLMs (positive class),
from the confusion matrices (evaluation vs Gold Standard).
"""

import pandas as pd

def compute_metrics(tn, fp, fn, tp):
    """Compute positive-class and macro metrics."""
    # Positive class (oui = 1)
    prec_oui = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec_oui = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_oui = 2 * prec_oui * rec_oui / (prec_oui + rec_oui) if (prec_oui + rec_oui) > 0 else 0

    # Negative class (non = 0)
    prec_non = tn / (tn + fn) if (tn + fn) > 0 else 0
    rec_non = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1_non = 2 * prec_non * rec_non / (prec_non + rec_non) if (prec_non + rec_non) > 0 else 0

    # MACRO = mean of the two classes
    precision_macro = (prec_oui + prec_non) / 2
    recall_macro = (rec_oui + rec_non) / 2
    f1_macro = (f1_oui + f1_non) / 2

    accuracy = (tp + tn) / (tp + tn + fp + fn)

    # Yes rate (for the paper)
    total = tp + tn + fp + fn
    yes_rate = (tp + fp) / total

    return prec_oui, rec_oui, f1_oui, prec_non, rec_non, f1_non, precision_macro, recall_macro, f1_macro, accuracy, yes_rate


def compute_mcc(tn, fp, fn, tp):
    """Compute the Matthews Correlation Coefficient."""
    num = (tp * tn) - (fp * fn)
    den = ((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) ** 0.5
    return num / den if den > 0 else 0


# Data extracted from the logs (confusion matrices vs GOLD_STANDARD)
# Format: (TN, FP, FN, TP) - pred_non/true_non, pred_oui/true_non, pred_non/true_oui, pred_oui/true_oui
models_data = {
    "Qwen2.5-7B-Instruct": (329, 236, 136, 314),
    "Qwen2.5-32B-Instruct": (269, 296, 81, 369),
    "Qwen3-32B-Thinking": (114, 451, 39, 411),  # Reasoning model with thinking mode
    "Llama-3.1-8B-Instruct": (499, 66, 293, 157),
    "Llama-3.1-70B-Instruct": (138, 427, 27, 423),
    "Mistral-Nemo-Instruct": (184, 381, 66, 384),
    "Gemma-2-27B-it": (54, 511, 7, 443),
    "Command-R-35B": (25, 540, 13, 437),
    "Aya-Expanse-32B": (43, 522, 18, 432),
    "Saul-7B-Instruct": (34, 531, 21, 429),
}

# Compute metrics
results = []
for model_name, (tn, fp, fn, tp) in models_data.items():
    prec_oui, rec_oui, f1_oui, prec_non, rec_non, f1_non, prec_macro, rec_macro, f1_macro, accuracy, yes_rate = compute_metrics(tn, fp, fn, tp)
    mcc = compute_mcc(tn, fp, fn, tp)

    results.append({
        "Model": model_name,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Yes%": yes_rate,
        # OUI class (positive)
        "P_oui": prec_oui,
        "R_oui": rec_oui,
        "F1_oui": f1_oui,
        # NON class
        "P_non": prec_non,
        "R_non": rec_non,
        "F1_non": f1_non,
        # MACRO
        "P_macro": prec_macro,
        "R_macro": rec_macro,
        "F1_macro": f1_macro,
        # Global
        "Accuracy": accuracy,
        "MCC": mcc,
    })

# Build the DataFrame and sort by descending MCC
df = pd.DataFrame(results)
df = df.sort_values("MCC", ascending=False).reset_index(drop=True)

# ============================================================
# DISPLAY
# ============================================================
print("=" * 80)
print("POSITIVE-CLASS (OUI) METRICS FOR THE 10 LLMs (vs Gold Standard)")
print("=" * 80)
print()

# POSITIVE-CLASS table (for the paper)
print("📊 POSITIVE-CLASS (OUI) Metrics:\n")
print(f"{'Model':<25} {'Yes%':>6} {'P':>7} {'R':>7} {'F1':>7} {'Acc':>7} {'MCC':>7}")
print("-" * 70)
for _, row in df.iterrows():
    print(f"{row['Model']:<25} {row['Yes%']:>6.2%} {row['P_oui']:>7.2%} {row['R_oui']:>7.2%} {row['F1_oui']:>7.2%} {row['Accuracy']:>7.2%} {row['MCC']:>7.4f}")

print()
print()

# MACRO table for comparison
print("📊 MACRO Metrics (for reference):\n")
print(f"{'Model':<25} {'P_macro':>7} {'R_macro':>7} {'F1_macro':>7}")
print("-" * 50)
for _, row in df.iterrows():
    print(f"{row['Model']:<25} {row['P_macro']:>7.2%} {row['R_macro']:>7.2%} {row['F1_macro']:>7.2%}")

print()
print()

# Comparison F1_oui vs F1_macro
print("📊 Comparison F1_oui vs F1_macro:\n")
print(f"{'Model':<25} {'F1_oui':>10} {'F1_non':>10} {'F1_macro':>10}")
print("-" * 58)
for _, row in df.iterrows():
    print(f"{row['Model']:<25} {row['F1_oui']:>10.2%} {row['F1_non']:>10.2%} {row['F1_macro']:>10.2%}")

print()
print("-" * 80)
print("Confusion matrices:")
print("-" * 80)
print(df[["Model", "TP", "TN", "FP", "FN"]].to_string(index=False))

# ============================================================
# LATEX TABLE (for the paper) - POSITIVE CLASS
# ============================================================
print()
print()
print("=" * 80)
print("LATEX TABLE (POSITIVE-CLASS METRICS)")
print("=" * 80)
print()

print(r"""\begin{table}[t]
\centering
\small
\begin{tabular}{lrrrrr}
\toprule
\textbf{Model} & \textbf{Yes\%} & \textbf{P} & \textbf{R} & \textbf{F1} & \textbf{MCC} \\
\midrule""")

for _, row in df.iterrows():
    model = row['Model'].replace("Instruct", "").replace("-it", "").strip()
    print(f"{model} & {row['Yes%']:.2f} & {row['P_oui']:.2f} & {row['R_oui']:.2f} & {row['F1_oui']:.2f} & {row['MCC']:.2f} \\\\")

print(r"""\bottomrule
\end{tabular}
\caption{Zero-shot LLM results ($n=1{,}015$, positive class metrics).}
\label{tab:zeroshot}
\end{table}""")

# ============================================================
# SUMMARY
# ============================================================
print()
print()
print("=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Best MCC:      {df.iloc[0]['Model']} ({df.iloc[0]['MCC']:.4f})")
print(f"Best F1_oui:   {df.loc[df['F1_oui'].idxmax()]['Model']} ({df['F1_oui'].max():.4f})")
print(f"Best F1_macro: {df.loc[df['F1_macro'].idxmax()]['Model']} ({df['F1_macro'].max():.4f})")
print(f"Best Accuracy: {df.loc[df['Accuracy'].idxmax()]['Model']} ({df['Accuracy'].max():.4f})")

# Per-annotator metrics

In [ ]:
"""
F1 and accuracy for 10 LLMs (positive class).
Evaluation vs Gold Standard, A1, and A2.
"""

import pandas as pd

def compute_metrics(tn, fp, fn, tp):
    """Compute positive-class and macro metrics."""
    # Positive class (oui = 1)
    prec_oui = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec_oui = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_oui = 2 * prec_oui * rec_oui / (prec_oui + rec_oui) if (prec_oui + rec_oui) > 0 else 0

    # Negative class (non = 0)
    prec_non = tn / (tn + fn) if (tn + fn) > 0 else 0
    rec_non = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1_non = 2 * prec_non * rec_non / (prec_non + rec_non) if (prec_non + rec_non) > 0 else 0

    # MACRO = mean of the two classes
    precision_macro = (prec_oui + prec_non) / 2
    recall_macro = (rec_oui + rec_non) / 2
    f1_macro = (f1_oui + f1_non) / 2

    accuracy = (tp + tn) / (tp + tn + fp + fn)

    # Yes rate
    total = tp + tn + fp + fn
    yes_rate = (tp + fp) / total

    return prec_oui, rec_oui, f1_oui, prec_non, rec_non, f1_non, precision_macro, recall_macro, f1_macro, accuracy, yes_rate


def compute_mcc(tn, fp, fn, tp):
    """Compute the Matthews Correlation Coefficient."""
    num = (tp * tn) - (fp * fn)
    den = ((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) ** 0.5
    return num / den if den > 0 else 0


# ============================================================
# DATA: format (TN, FP, FN, TP)
# ============================================================

# VS GOLD STANDARD
models_gold = {
    "Qwen2.5-7B": (329, 236, 136, 314),
    "Qwen2.5-32B": (269, 296, 81, 369),
    "Qwen3-32B-Thinking": (114, 451, 39, 411),
    "LLaMA-3.1-8B": (499, 66, 293, 157),
    "LLaMA-3.1-70B": (138, 427, 27, 423),
    "Mistral-Nemo-12B": (184, 381, 66, 384),
    "Gemma-2-27B": (54, 511, 7, 443),
    "Command-R-35B": (25, 540, 13, 437),
    "Aya-Expanse-32B": (43, 522, 18, 432),
    "SAUL-7B": (34, 531, 21, 429),
}

# VS A1
models_a1 = {
    "Qwen2.5-7B": (297, 210, 168, 340),
    "Qwen2.5-32B": (247, 260, 103, 405),
    "Qwen3-32B-Thinking": (107, 400, 46, 462),
    "LLaMA-3.1-8B": (443, 64, 349, 159),
    "LLaMA-3.1-70B": (127, 380, 38, 470),
    "Mistral-Nemo-12B": (165, 342, 85, 423),
    "Gemma-2-27B": (48, 459, 13, 495),
    "Command-R-35B": (23, 484, 15, 493),
    "Aya-Expanse-32B": (39, 468, 22, 486),
    "SAUL-7B": (32, 475, 23, 485),
}

# VS A2
models_a2 = {
    "Qwen2.5-7B": (195, 139, 270, 411),
    "Qwen2.5-32B": (174, 160, 176, 505),
    "Qwen3-32B-Thinking": (74, 260, 79, 602),
    "LLaMA-3.1-8B": (292, 42, 500, 181),
    "LLaMA-3.1-70B": (85, 249, 80, 601),
    "Mistral-Nemo-12B": (115, 219, 135, 546),
    "Gemma-2-27B": (39, 295, 22, 659),
    "Command-R-35B": (16, 318, 22, 659),
    "Aya-Expanse-32B": (21, 313, 40, 641),
    "SAUL-7B": (25, 309, 30, 651),
}


def compute_all_metrics(models_data, target_name):
    """Compute metrics for all models."""
    results = []
    for model_name, (tn, fp, fn, tp) in models_data.items():
        prec_oui, rec_oui, f1_oui, prec_non, rec_non, f1_non, prec_macro, rec_macro, f1_macro, accuracy, yes_rate = compute_metrics(tn, fp, fn, tp)
        mcc = compute_mcc(tn, fp, fn, tp)

        results.append({
            "Model": model_name,
            "Yes%": yes_rate,
            # Positive class (oui)
            "P_oui": prec_oui,
            "R_oui": rec_oui,
            "F1_oui": f1_oui,
            # Negative class (non)
            "P_non": prec_non,
            "R_non": rec_non,
            "F1_non": f1_non,
            # MACRO
            "P_macro": prec_macro,
            "R_macro": rec_macro,
            "F1_macro": f1_macro,
            # Global
            "Accuracy": accuracy,
            "MCC": mcc,
        })

    df = pd.DataFrame(results)
    df = df.sort_values("MCC", ascending=False).reset_index(drop=True)
    return df


def print_latex_table(df, target_name, label):
    """Generate the LaTeX table with positive-class metrics."""
    print(f"""
\\begin{{table}}[h]
\\centering
\\small
\\begin{{tabular}}{{lccccc}}
\\toprule
\\textbf{{Model}} & \\textbf{{Yes\\%}} & \\textbf{{P}} & \\textbf{{R}} & \\textbf{{F1}} & \\textbf{{MCC}} \\\\
\\midrule""")

    for _, row in df.iterrows():
        model = row['Model']
        # Add annotations
        if model == "LLaMA-3.1-70B":
            model += "$^\\dagger$"
        elif model == "Qwen3-32B-Thinking":
            model = "Qwen3-32B$^\\ddagger$"

        print(f"{model} & .{round(row['Yes%']*100):02d} & .{round(row['P_oui']*100):02d} & .{round(row['R_oui']*100):02d} & .{round(row['F1_oui']*100):02d} & .{round(row['MCC']*100):02d} \\\\")

    print(f"""\\bottomrule
\\end{{tabular}}
\\caption{{Zero-shot results vs.\\ {target_name} (positive class). P = precision, R = recall. $^\\dagger$4-bit quantization. $^\\ddagger$Reasoning model.}}
\\label{{tab:{label}}}
\\end{{table}}
""")


def print_console_table(df, title):
    """Print the console table with positive-class metrics."""
    print(f"\n{'Model':<22} {'Yes%':>6} {'P':>7} {'R':>7} {'F1':>7} {'Acc':>7} {'MCC':>7}")
    print("-" * 70)
    for _, row in df.iterrows():
        print(f"{row['Model']:<22} {row['Yes%']:>6.2%} {row['P_oui']:>7.2%} {row['R_oui']:>7.2%} {row['F1_oui']:>7.2%} {row['Accuracy']:>7.2%} {row['MCC']:>7.4f}")


def print_macro_comparison(df):
    """Print the comparison with macro metrics."""
    print(f"\n📊 Comparison F1_oui vs F1_macro:\n")
    print(f"{'Model':<22} {'F1_oui':>10} {'F1_non':>10} {'F1_macro':>10}")
    print("-" * 55)
    for _, row in df.iterrows():
        print(f"{row['Model']:<22} {row['F1_oui']:>10.2%} {row['F1_non']:>10.2%} {row['F1_macro']:>10.2%}")


# ============================================================
# COMPUTE AND DISPLAY
# ============================================================

print("=" * 80)
print("GOLD STANDARD - POSITIVE CLASS (OUI)")
print("=" * 80)
df_gold = compute_all_metrics(models_gold, "Gold")
print_console_table(df_gold, "Gold")
print_macro_comparison(df_gold)
print_latex_table(df_gold, "gold ($n=1{,}015$)", "zeroshot-gold")


print("\n" + "=" * 80)
print("A1 - POSITIVE CLASS (OUI)")
print("=" * 80)
df_a1 = compute_all_metrics(models_a1, "A1")
print_console_table(df_a1, "A1")
print_macro_comparison(df_a1)
print_latex_table(df_a1, "A\\textsubscript{1}", "zeroshot-a1")


print("\n" + "=" * 80)
print("A2 - POSITIVE CLASS (OUI)")
print("=" * 80)
df_a2 = compute_all_metrics(models_a2, "A2")
print_console_table(df_a2, "A2")
print_macro_comparison(df_a2)
print_latex_table(df_a2, "A\\textsubscript{2}", "zeroshot-a2")


print("\n" + "=" * 80)
print("SUMMARY - POSITIVE CLASS")
print("=" * 80)
print(f"Gold - Best MCC: {df_gold.iloc[0]['Model']} ({df_gold.iloc[0]['MCC']:.4f}), Best F1_oui: {df_gold.loc[df_gold['F1_oui'].idxmax()]['Model']} ({df_gold['F1_oui'].max():.4f})")
print(f"A1   - Best MCC: {df_a1.iloc[0]['Model']} ({df_a1.iloc[0]['MCC']:.4f}), Best F1_oui: {df_a1.loc[df_a1['F1_oui'].idxmax()]['Model']} ({df_a1['F1_oui'].max():.4f})")
print(f"A2   - Best MCC: {df_a2.iloc[0]['Model']} ({df_a2.iloc[0]['MCC']:.4f}), Best F1_oui: {df_a2.loc[df_a2['F1_oui'].idxmax()]['Model']} ({df_a2['F1_oui'].max():.4f})")